In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for all envs)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os # Import the os module

# 1. LOAD DATA
print("=" * 65)
print("  HEART DISEASE CLASSIFICATION — FULL ML PIPELINE")
print("=" * 65)

try:
    df = pd.read_csv("heart.csv")
    print(f"\n✅  Loaded heart.csv  →  {df.shape[0]} rows × {df.shape[1]} columns")
except FileNotFoundError:
    # ── Synthetic fallback so the script always runs ──
    print("\n⚠️  heart.csv not found — generating synthetic data for demo …")
    from sklearn.datasets import make_classification
    np.random.seed(42)
    n = 500
    X_raw, y_raw = make_classification(n_samples=n, n_features=10,
                                       n_informative=7, random_state=42)
    df = pd.DataFrame(X_raw, columns=[f"feat_{i}" for i in range(10)])

    # Inject a few categorical-style columns to demonstrate encoding
    df["sex"]   = np.random.choice(["M", "F"], n)
    df["cp"]    = np.random.choice(["typical", "atypical", "non-anginal", "asymptomatic"], n)
    df["fbs"]   = np.random.choice([0, 1], n)
    df["target"] = y_raw
    print(f"   Generated synthetic dataset  →  {df.shape[0]} rows × {df.shape[1]} columns")

print("\n── Raw data sample ──")
print(df.head(3).to_string())
print(f"\nColumn dtypes:\n{df.dtypes.to_string()}")

# ──────────────────────────────────────────────────────────────────────────────
# 2. IDENTIFY COLUMNS
# ──────────────────────────────────────────────────────────────────────────────
target_col = "HeartDisease"
feature_cols = [c for c in df.columns if c != target_col]

# Separate object columns (text) from numeric ones
text_cols    = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()
numeric_cols = df[feature_cols].select_dtypes(exclude=["object"]).columns.tolist()

print(f"\n── Column inventory ──")
print(f"  Text columns    : {text_cols  if text_cols   else 'None'}")
print(f"  Numeric columns : {numeric_cols[:5]}{'…' if len(numeric_cols) > 5 else ''}")
print(f"  Target column   : {target_col}")

# ──────────────────────────────────────────────────────────────────────────────
# 3. ENCODING
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 1 — ENCODING")
print("=" * 65)

df_encoded = df.copy()
label_encoders = {}

if text_cols:
    # Decide which columns get Label Encoding vs One-Hot Encoding
    # Heuristic: binary text cols → Label Encoding; multi-class → One-Hot
    le_cols  = [c for c in text_cols if df[c].nunique() == 2]
    ohe_cols = [c for c in text_cols if df[c].nunique()  > 2]

    # ── Label Encoding ──
    if le_cols:
        print(f"\n🔢  Label Encoding  →  {le_cols}")
        for col in le_cols:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col])
            label_encoders[col] = le
            print(f"    {col}: classes = {list(le.classes_)}")

    # ── One-Hot Encoding ──
    if ohe_cols:
        print(f"\n🔡  One-Hot Encoding  →  {ohe_cols}")
        df_encoded = pd.get_dummies(df_encoded, columns=ohe_cols, drop_first=True)
        new_ohe_cols = [c for c in df_encoded.columns
                        if any(c.startswith(o + "_") for o in ohe_cols)]
        print(f"    New columns created: {new_ohe_cols}")
else:
    print("\n  No text columns detected — skipping encoding step.")

# Refresh feature list after encoding
feature_cols_enc = [c for c in df_encoded.columns if c != target_col]
print(f"\n  Total features after encoding: {len(feature_cols_enc)}")

# ──────────────────────────────────────────────────────────────────────────────
# 4. TRAIN / TEST SPLIT
# ──────────────────────────────────────────────────────────────────────────────
X = df_encoded[feature_cols_enc].values
y = df_encoded[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\n  Train size: {X_train.shape[0]}   Test size: {X_test.shape[0]}")

# ──────────────────────────────────────────────────────────────────────────────
# 5. SCALING
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 2 — STANDARD SCALING")
print("=" * 65)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("  ✅  StandardScaler applied (fit on train, transform on test)")

# ──────────────────────────────────────────────────────────────────────────────
# 6. HELPER — train & evaluate a model
# ──────────────────────────────────────────────────────────────────────────────
def evaluate(name, model, Xtr, Xte, ytr, yte, cv_folds=5):
    model.fit(Xtr, ytr)
    y_pred   = model.predict(Xte)
    acc      = accuracy_score(yte, y_pred)
    cv_acc   = cross_val_score(model, Xtr, ytr, cv=cv_folds, scoring="accuracy").mean()
    return {
        "name"    : name,
        "model"   : model,
        "test_acc": acc,
        "cv_acc"  : cv_acc,
        "y_pred"  : y_pred,
    }

# ──────────────────────────────────────────────────────────────────────────────
# 7. CLASSIFICATION — WITHOUT PCA
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 3 — CLASSIFICATION (without PCA)")
print("=" * 65)

models_cfg = {
    "SVM"              : SVC(kernel="rbf", C=1.0, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"    : RandomForestClassifier(n_estimators=100, random_state=42),
}

results_no_pca = []
for mname, mobj in models_cfg.items():
    r = evaluate(mname, mobj, X_train_sc, X_test_sc, y_train, y_test)
    results_no_pca.append(r)
    print(f"\n  [{mname}]")
    print(f"    Test Accuracy : {r['test_acc']:.4f}  ({r['test_acc']*100:.2f}%)")
    print(f"    CV  Accuracy  : {r['cv_acc']:.4f}  ({r['cv_acc']*100:.2f}%)")
    print(classification_report(y_test, r["y_pred"], zero_division=0))

best_no_pca = max(results_no_pca, key=lambda x: x["test_acc"])
print(f"\n🏆  Best model (no PCA): {best_no_pca['name']}  "
      f"— Accuracy = {best_no_pca['test_acc']*100:.2f}%")

print("\n" + "=" * 65)
print("  STEP 4 — PCA DIMENSIONALITY REDUCTION")
print("=" * 65)

pca_full = PCA().fit(X_train_sc)
explained = np.cumsum(pca_full.explained_variance_ratio_)

n_components_95 = int(np.searchsorted(explained, 0.95) + 1)
print(f"\n  Original dimensions : {X_train_sc.shape[1]}")
print(f"  Components for 95%  : {n_components_95}")

pca = PCA(n_components=n_components_95, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)
print(f"  Reduced shape (train): {X_train_pca.shape}")

print("\n" + "=" * 65)
print("  STEP 5 — CLASSIFICATION (with PCA)")
print("=" * 65)

models_cfg_pca = {
    "SVM"              : SVC(kernel="rbf", C=1.0, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"    : RandomForestClassifier(n_estimators=100, random_state=42),
}

results_pca = []
for mname, mobj in models_cfg_pca.items():
    r = evaluate(mname, mobj, X_train_pca, X_test_pca, y_train, y_test)
    results_pca.append(r)
    print(f"\n  [{mname}] with PCA")
    print(f"    Test Accuracy : {r['test_acc']:.4f}  ({r['test_acc']*100:.2f}%)")
    print(f"    CV  Accuracy  : {r['cv_acc']:.4f}  ({r['cv_acc']*100:.2f}%)")

best_pca = max(results_pca, key=lambda x: x["test_acc"])
print(f"\n🏆  Best model (with PCA): {best_pca['name']}  "
      f"— Accuracy = {best_pca['test_acc']*100:.2f}%")

print("\n" + "=" * 65)
print("  FINAL COMPARISON SUMMARY")
print("=" * 65)

summary_rows = []
for r_no, r_pc in zip(results_no_pca, results_pca):
    delta = r_pc["test_acc"] - r_no["test_acc"]
    summary_rows.append({
        "Model"              : r_no["name"],
        "Acc (no PCA)"       : f"{r_no['test_acc']*100:.2f}%",
        "Acc (with PCA)"     : f"{r_pc['test_acc']*100:.2f}%",
        "Δ Accuracy"         : f"{delta*100:+.2f}%",
    })

summary_df = pd.DataFrame(summary_rows)
print(f"\n{summary_df.to_string(index=False)}")
print(f"\n  Dimensions before PCA : {X_train_sc.shape[1]}")
print(f"  Dimensions after  PCA : {n_components_95}  (95% variance retained)")
print("\n  💡 Trade-off insight:")
print("     PCA reduces feature dimensions → faster training & less overfitting.")
print("     A small accuracy dip is expected; the gain is speed & generalization.")

plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Heart Disease Classification — Pipeline Results",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

model_names = [r["name"] for r in results_no_pca]
acc_no_pca  = [r["test_acc"] for r in results_no_pca]
acc_pca     = [r["test_acc"] for r in results_pca]
x_idx       = np.arange(len(model_names))
bar_w       = 0.35
colors      = ["#2196F3", "#FF5722"]

ax1 = fig.add_subplot(gs[0, :2])
b1 = ax1.bar(x_idx - bar_w/2, acc_no_pca, bar_w, label="Without PCA", color=colors[0], alpha=0.85)
b2 = ax1.bar(x_idx + bar_w/2, acc_pca,    bar_w, label="With PCA",    color=colors[1], alpha=0.85)
ax1.set_xticks(x_idx); ax1.set_xticklabels(model_names, fontsize=11)
ax1.set_ylabel("Test Accuracy"); ax1.set_title("Model Accuracy: With vs Without PCA")
ax1.set_ylim(0, 1.12)
ax1.legend()
for bar in b1: ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                         f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in b2: ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                         f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(range(1, len(explained)+1), explained, marker="o", color="#4CAF50", linewidth=2)
ax2.axhline(0.95, color="red", linestyle="--", label="95% threshold")
ax2.axvline(n_components_95, color="orange", linestyle="--",
            label=f"n={n_components_95} components")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Explained Variance")
ax2.set_title("PCA — Explained Variance Curve")
ax2.legend(fontsize=8)

for i, r in enumerate(results_no_pca):
    ax = fig.add_subplot(gs[1, i])
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"{r['name']}\n(no PCA) Acc={r['test_acc']:.3f}", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

# Create the directory if it doesn't exist
output_dir = "/mnt/user-data/outputs/"
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, "heart_pipeline_results.png"),
            dpi=150, bbox_inches="tight")
print("\n  📊  Visualisation saved → heart_pipeline_results.png")
print("\n✅  Pipeline complete.")

  HEART DISEASE CLASSIFICATION — FULL ML PIPELINE

✅  Loaded heart.csv  →  918 rows × 12 columns

── Raw data sample ──
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR ExerciseAngina  Oldpeak ST_Slope  HeartDisease
0   40   M           ATA        140          289          0     Normal    172              N      0.0       Up             0
1   49   F           NAP        160          180          0     Normal    156              N      1.0     Flat             1
2   37   M           ATA        130          283          0         ST     98              N      0.0       Up             0

Column dtypes:
Age                 int64
Sex                object
ChestPainType      object
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG         object
MaxHR               int64
ExerciseAngina     object
Oldpeak           float64
ST_Slope           object
HeartDisease        int64

── Column inventory ──
  Text columns    : ['Sex', 